# CGAS Deliverable 1: top 250 Indian ingredients from the dish dataset

Sarthak Aggarwal (MT26139) and Divyanshu Joshi (MT26120), CGAS Group 26, Deliverable 1

Source: `D1_input_dishes_725.zip`, which holds `Indian_Food_Ingredients_Nutrition_CookingMethods.csv` (725 dishes).

This notebook builds the first list of ingredient names from the dishes. After it, the list was edited by hand and more ingredients were added from the 10,000 recipes (see the README). The final data is in `D1_0_ingredient_250_final_dataset.csv`.

**Proof of hand-cleaning:** the cleaning rules below were decided by hand. Each rule is written out, and the code applies exactly those rules. Every changed name shows its rule ID.

In [ ]:
# Sarthak Aggarwal MT26139
# Divyanshu Joshi MT26120
import csv, io, re, zipfile
from collections import Counter, defaultdict

DISH_ZIP = "D1_input_dishes_725.zip"
DISH_CSV = "Indian_Food_Ingredients_Nutrition_CookingMethods.csv"   # the file inside the zip
TOP_N = 250

## Step 1: keep only Indian-labelled dishes

In [ ]:
NON_INDIAN = {
    "Continental", "Italian Recipes", "Fusion", "Chinese", "Indo Chinese", "Mexican", "Thai",
    "Asian", "European", "Mediterranean", "Caribbean", "African", "Middle Eastern", "Greek",
    "Japanese", "Vietnamese", "Malaysian", "Hunan",
}
with zipfile.ZipFile(DISH_ZIP) as z, z.open(DISH_CSV) as raw:
    dishes = list(csv.DictReader(io.TextIOWrapper(raw, encoding="utf-8")))
indian = [d for d in dishes if d["Cuisine"].strip() not in NON_INDIAN]
print("all dishes:", len(dishes), "| Indian-labelled:", len(indian))

## Step 2: cleaning rules (decided by hand)

| Rule | Decision |
|---|---|
| R1 | Red chilli powder is a separate ingredient. Fresh red chilies and dried red chilies are two more separate ingredients. |
| R2 | Spelling is "chilies" (plural) for whole pods. "green chilli" is removed because "green chillies" already exists. |
| R3 | "cloves garlic" is written as "garlic". |
| R4 | Cumin seeds and cumin powder are different ingredients. |
| R5 | Garam masala powder is deleted. |
| R6 | Cinnamon stick and cinnamon powder are different ingredients. |
| R7 | The English name is "ghee". |
| R8 | The English name is "curd". Its Roman variants include "yogurt" and "dahi". "yogurt" rows are merged into "curd". |
| R9 | Ginger garlic paste is deleted. |
| R10 | A bracketed name is kept as a Roman variant only if it is an approved Hindi or Sanskrit name (the list `HINDI_ROMAN_OK` below). Other brackets are descriptions and are dropped. A bracket that names a different ingredient in the table (for example `ghee`) is also dropped. |
| R11 | **Duplication check (no renaming).** After cleaning, every word that appears in 2 or more of the top-250 names is printed in Step 3b. Each group is then judged by hand. This is how R12 to R14 were found. |
| R12 | **Leftover recipe words and typos are removed from the name.** A word repeated twice in a row is collapsed (`salt salt` becomes `salt`), except real names like `chow chow` (list `KEEP_REPEATS`). A fixed list `JUNK_FIX` handles the rest (`sunflower oil pan`, `ghee roasting`, `water knead`, `chicken bs`, ...). `cardamom podsseeds` is written `cardamom pods` (two words had been glued together in the source). |
| R13 | **Same ingredient, different wording: merged.** `potatoes` into `potato`, `black pepper corns` into `black peppercorns`, `mint leaves` into `mint`, `methi leaves bunch` into `methi leaves`, `fish fillet` into `fish`, `bay leaves` into `bay leaf`. The list is `MERGE`. The Roman variant (for example `pudina`) moves to the merged name. |
| R14 | **Too vague to be an ingredient: deleted.** `green`, `oil`, and `flour ( flour)` (the bracket repeats the word, so nothing is known). `flour (maida)` is not deleted. It is renamed `refined flour` with Roman variant `maida`. |

**Looked at and kept separate on purpose (R11 decisions):**
- `red chilli powder` and `kashmiri red chilli powder`: different variety (see R1).
- `curd` and `hung curd`: hung curd is a different product (water drained).
- `coriander`, `coriander leaves`, `coriander seeds`, `coriander powder`: different parts of the plant, used differently.
- `green chutney` and `green`: solved by deleting the vague `green` (R14).
- `chilli powder` and `red chilli powder`: a plain "chilli powder" row does not occur in the top 250, so nothing to merge.

R12 to R14 are my current decisions. They are plain Python dicts in the next cell, so any of them can be changed in one line and the notebook re-run.

Names not covered by a rule are left unchanged and marked `no rule yet`. More rules will be added as the list is checked.

In [ ]:
# raw name -> (english name or None if deleted, roman variant from brackets, rule ID)
CLEAN = {
    "red chilli powder": ("red chilli powder", "", "R1"),
    "kashmiri red chilli powder": ("kashmiri red chilli powder", "", "R1"),
    "green chillies": ("green chilies", "", "R2"),
    "green chilli": (None, "", "R2"),
    "dry red chillies": ("dried red chilies", "", "R1"),
    "dry red chilli": ("dried red chilies", "", "R1"),
    "dried red chilli": ("dried red chilies", "", "R1"),
    "kashmiri dry red chillies": ("dried kashmiri red chilies", "", "R1"),
    "byadagi dried chillies": ("dried byadagi chilies", "", "R1"),
    "guntur dried chillies": ("dried guntur chilies", "", "R1"),
    "red chilli": ("fresh red chilies", "", "R1"),
    "cloves garlic": ("garlic", "", "R3"),
    "garlic": ("garlic", "", "R3"),
    "cumin seeds (jeera)": ("cumin seeds", "jeera", "R4"),
    "cumin seeds": ("cumin seeds", "", "R4"),
    "cumin (jeera) seeds": ("cumin seeds", "jeera", "R4"),
    "cumin powder (jeera)": ("cumin powder", "jeera", "R4"),
    "cumin powder": ("cumin powder", "", "R4"),
    "cumin powder take out": ("cumin powder", "", "R4"),
    "garam masala powder": (None, "", "R5"),
    "cinnamon stick (dalchini)": ("cinnamon stick", "dalchini", "R6"),
    "cinnamon stick": ("cinnamon stick", "", "R6"),
    "cinnamon powder (dalchini)": ("cinnamon powder", "dalchini", "R6"),
    "cinnamon powder": ("cinnamon powder", "", "R6"),
    "ghee": ("ghee", "", "R7"),
    "curd": ("curd", "yogurt|dahi", "R8"),
    "yogurt": ("curd", "", "R8"),
    "ginger garlic paste": (None, "", "R9"),
}

HINDI_ROMAN_OK = {
    "jeera", "dalchini", "laung", "dhania", "hing", "elaichi", "haldi", "adrak", "lehsun",
    "pyaz", "hari mirch", "lal mirch", "kali mirch", "ghee", "dahi", "doodh", "paneer",
    "chawal", "atta", "besan", "cheeni", "gud", "namak", "tel", "sarson", "rai", "methi",
    "pudina", "kadi patta", "nimbu", "tamatar", "gajar", "matar", "bhindi", "karela", "lauki",
    "chana dal", "toor dal", "moong dal", "urad dal", "masoor dal", "rajma", "chole", "poha",
    "rava", "suji", "maida", "khoya", "malai", "imli", "kokum", "saunf", "ajwain", "kesar",
    "badam", "kaju", "pista", "kishmish", "tej patta", "aloo", "dhaniya", "jaiphal",
    "shahi jeera", "mirchi", "elachi", "dalchini",
}

# R12: leftover recipe words / typos -> correct name
JUNK_FIX = {
    "salt per": "salt",
    "sunflower oil pan": "sunflower oil",
    "sunflower oil grease": "sunflower oil",
    "ghee roasting": "ghee",
    "ghee the parathas": "ghee",
    "wheat berries wheat": "wheat berries",
    "water knead": "water",
    "cashew nuts water": "cashew nuts",
    "chicken bs": "chicken",
    "tamarind marble": "tamarind",
    "cardamom podsseeds": "cardamom pods",
}

# R13: same ingredient, different wording -> one name
MERGE = {
    "potatoes": "potato",
    "black pepper corns": "black peppercorns",
    "mint leaves": "mint",
    "methi leaves bunch": "methi leaves",
    "fish fillet": "fish",
    "bay leaves": "bay leaf",
}

# R14: too vague, deleted
DELETE = {"green", "oil", "flour"}

# raw names that need their own answer (checked before everything else)
CLEAN_EXTRA = {
    "flour (maida)": ("refined flour", "maida", "R14"),
    "flour ( flour)": (None, "", "R14"),
}

OTHER_INGREDIENTS = {v[0] for v in CLEAN.values() if v[0]}

def add_rule(old, new):
    return new if old == "no rule yet" else old + ", " + new

# real names that repeat a word on purpose
KEEP_REPEATS = {"chow chow"}

def collapse_repeats(name):
    # "salt salt" -> "salt"
    if name in KEEP_REPEATS:
        return name
    words = name.split()
    return " ".join(w for i, w in enumerate(words) if i == 0 or w != words[i - 1])

def clean_name(raw, extra=True):
    """extra=False gives the old R1-R10 result, so R12-R14 can be compared against it."""
    if extra and raw in CLEAN_EXTRA:
        return CLEAN_EXTRA[raw]
    if raw in CLEAN:
        return CLEAN[raw]
    m = re.match(r"^(.*?)\s*\((.*?)\)\s*(.*)$", raw)
    if m:
        english = (m.group(1) + " " + m.group(3)).strip()
        inside = m.group(2).strip().lower()
        if inside in HINDI_ROMAN_OK and inside not in OTHER_INGREDIENTS:
            roman, rule = inside, "R10"
        else:
            roman, rule = "", "R10 bracket dropped"
    else:
        english, roman, rule = raw, "", "no rule yet"
    if not extra:
        return english, roman, rule
    fixed = JUNK_FIX.get(collapse_repeats(english), collapse_repeats(english))
    if fixed != english:
        english, rule = fixed, add_rule(rule, "R12")
    if english in DELETE:
        return None, "", "R14"
    if english in MERGE:
        return MERGE[english], roman, add_rule(rule, "R13")
    return english, roman, rule

def parse_list(text):
    return [x.strip().lower() for x in text.split(",") if x.strip()]


## Step 3: count cleaned ingredients per Indian dish and take the top 250
Counts use the cleaned names, so merged rows (for example yogurt into curd) are not double-counted.
`count_ingredients(extra=False)` is the old run with only R1 to R10. `count_ingredients(extra=True)` is the current run with R12 to R14 added. Both are kept so the effect of the new rules can be shown.

In [ ]:
def count_ingredients(extra):
    counts = Counter()
    roman_for = defaultdict(Counter)   # english -> Counter of Roman variants
    rule_for = defaultdict(set)
    for d in indian:
        seen = set()
        for raw in parse_list(d["Cleaned-Ingredients"]):
            english, roman, rule = clean_name(raw, extra)
            if english is None:
                continue
            if english not in seen:
                seen.add(english)
                counts[english] += 1
            for r in roman.split("|"):
                if r:
                    roman_for[english][r] += 1
            rule_for[english].add(rule)
    return counts, roman_for, rule_for

counts_old, _, _ = count_ingredients(extra=False)
counts, roman_for, rule_for = count_ingredients(extra=True)
top_old = counts_old.most_common(TOP_N)
top = counts.most_common(TOP_N)
print("distinct names before R12-R14:", len(counts_old), "| after:", len(counts))
print("top kept:", len(top))
print(top[:5])

## Step 3b: R11 duplication check
Rule R11: print every word that is shared by 2 or more names in the top 250. A shared word is not always a mistake (`powder`, `seeds`, `oil` are normal). Each group is read by hand. A plural is counted as the same word (`potato` and `potatoes`) by the small helper `singular()`. It is used only for this check and never changes a name in the output.

First the list of every name that R12 to R14 changed, with the number of dishes affected. Then the groups that are still shared after cleaning, for review.

In [ ]:
def singular(w):
    # only used to compare words in R11, never to rename
    if w.endswith("oes"):
        return w[:-2]
    if w.endswith("ss") or len(w) <= 3:
        return w
    return w[:-1] if w.endswith("s") else w

def shared_words(names):
    groups = defaultdict(list)
    for n in names:
        for w in {singular(x) for x in n.split()}:
            groups[w].append(n)
    return {w: v for w, v in groups.items() if len(v) >= 2}

# what R12-R14 changed: old name -> new name (None = deleted)
change_log = Counter()
for d in indian:
    for raw in parse_list(d["Cleaned-Ingredients"]):
        before = clean_name(raw, extra=False)[0]
        after, _, rule = clean_name(raw, extra=True)
        if before != after:
            change_log[(before, after, rule)] += 1
print("Changed by R12-R14 (dishes affected):")
for (before, after, rule), n in sorted(change_log.items(), key=lambda x: -x[1]):
    print(f"  {before!r:38} -> {after!r:24} {rule:8} {n}")

before_groups = shared_words([n for n, _ in top_old])
after_groups = shared_words([n for n, _ in top])
print("\nshared words before R12-R14:", len(before_groups), "| after:", len(after_groups))
print("\nStill shared after cleaning (review by hand):")
for w, names in sorted(after_groups.items()):
    print(f"  {w}: {', '.join(sorted(names))}")

## Step 4: Devanagari names
Only pairings that are reliable are filled in. Others are marked `manual`.

In [ ]:
DEVANAGARI = {
    "jeera": "जीरा", "hing": "हींग", "elaichi": "इलायची", "laung": "लौंग",
    "dalchini": "दालचीनी", "tej patta": "तेज पत्ता", "aloo": "आलू", "dhania": "धनिया",
    "haldi": "हल्दी", "adrak": "अदरक", "lehsun": "लहसुन", "pyaz": "प्याज़",
    "hari mirch": "हरी मिर्च", "lal mirch": "लाल मिर्च", "kali mirch": "काली मिर्च",
    "ghee": "घी", "dahi": "दही", "doodh": "दूध", "paneer": "पनीर", "chawal": "चावल",
    "atta": "आटा", "besan": "बेसन", "cheeni": "चीनी", "gud": "गुड़", "namak": "नमक",
    "tel": "तेल", "sarson": "सरसों", "rai": "राई", "methi": "मेथी", "dhaniya": "धनिया",
    "pudina": "पुदीना", "kadi patta": "कढ़ी पत्ता", "nimbu": "नींबू", "tamatar": "टमाटर",
    "gajar": "गाजर", "matar": "मटर", "bhindi": "भिंडी", "karela": "करेला", "lauki": "लौकी",
    "chana dal": "चना दाल", "toor dal": "तुअर दाल", "moong dal": "मूंग दाल", "urad dal": "उड़द दाल",
    "masoor dal": "मसूर दाल", "rajma": "राजमा", "chole": "छोले", "poha": "पोहा",
    "rava": "रवा", "suji": "सूजी", "maida": "मैदा", "khoya": "खोया", "malai": "मलाई",
    "imli": "इमली", "kokum": "कोकम", "saunf": "सौंफ", "ajwain": "अजवायन", "kesar": "केसर",
    "badam": "बादाम", "kaju": "काजू", "pista": "पिस्ता", "kishmish": "किशमिश",
}

def lookup_devanagari(english, romans):
    for k in [english.lower()] + sorted(romans):
        if k in DEVANAGARI:
            return DEVANAGARI[k]
    return ""


## Output

In [ ]:
rows = []
for rank, (english, c) in enumerate(top, start=1):
    romans = [r for r, _ in roman_for[english].most_common(2)]   # at most 2 variants
    dev_name = lookup_devanagari(english, romans)
    rules = sorted(r for r in rule_for[english] if r)
    if len(rules) > 1:
        rules = [r for r in rules if r != "no rule yet"]   # another rule applied to some spelling
    rule = ", ".join(rules)
    rows.append([rank, english, "|".join(romans), c, round(c / len(indian), 4),
                 dev_name, "" if dev_name else "manual", rule or "no rule yet"])

# quick checks before saving
names = [r[1] for r in rows]
assert len(names) == len(set(names)), "duplicate english name"
assert all(n == n.lower() for n in names), "name not lowercase"
assert all(len(r[2].split("|")) <= 2 for r in rows), "more than 2 roman variants"

with open("D1_1_first_list.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["rank", "english_name", "roman_variants", "dish_count", "share_of_indian_dishes",
                "hindi_devanagari", "status", "cleaning_rule"])
    w.writerows(rows)
print("saved D1_1_first_list.csv with", len(rows), "rows")
print("rows with no rule at all:", sum(1 for r in rows if r[7] == "no rule yet"))

## Errors found while cleaning (kept on purpose, so the mistakes are visible)

1. **Bracket text leaked into the Roman column.** In an earlier version of the CSV the Roman column held recipe notes such as `most preserved fish have salt so discretion using the salt` (on salt), `if not baking` (on sunflower oil) and `suit spice levels` (on red chilli powder). Cause: the bracket was copied without checking it. Fixed by R10 and by keeping at most 2 Roman variants per ingredient.
2. **Recipe words stuck to the name.** Examples: `salt per`, `sunflower oil pan`, `ghee the parathas`, `water knead`. They were counted as separate ingredients. Fixed by R12.
3. **Repeated words.** `salt salt`, `cloves cloves` (from `cloves cloves (laung)`), `sesame oil oil`. Fixed by R12.
4. **Two words glued together.** `cardamom podsseeds` (78 dishes). The space was lost in the source text, so it was fixed by hand in R12.
5. **Same ingredient counted twice.** `potato` and `potatoes`, `black pepper corns` and `black peppercorns`, `mint` and `mint leaves`. Found with R11 and fixed by R13.
6. **A rule that would have deleted real data.** Deleting the vague name `flour` would also have deleted `flour (maida)` (11 dishes), which is refined flour. It is now renamed `refined flour` (R14).
7. **Capital letters.** One row (`Garlic paste`) was not lowercase in the earlier CSV. All names are lowercased when the list is read.
8. **Stale output.** The earlier CSV had fewer rows than the 250 asked for. The CSV is now written again from this notebook, so every row comes from the rules above.

**What happened next:** this notebook only builds the first list. The names, Roman variants and Hindi names were then edited by hand, more ingredients were added from the 10,000 recipes, and the taste, smell and scientific-name columns were filled in. The result is `D1_0_ingredient_250_final_dataset.csv`.

## Repetition check on the final dataset (read only)

This cell only reads `D1_0_ingredient_250_final_dataset.csv`. It writes nothing. It checks for:
1. the same `english_name` twice (ignoring capitals and spaces)
2. the same words in a different order or as a plural (`potato` and `potatoes`)
3. almost-identical names, which are possible typos
4. one Roman variant listed under two different english names
5. the same variant written twice in one row

Check 4 is a list to read by hand. Some clashes are fine (for example `pudina` for both `mint` and `fresh peppermint leaves`), and others are mistakes.

In [ ]:
# Sarthak Aggarwal MT26139
# Divyanshu Joshi MT26120
# Repetition check on the final dataset. Read only: nothing here writes a file.
import re
import difflib
import pandas as pd

CSV_PATH = "D1_0_ingredient_250_final_dataset.csv"

def load_ingredients(path):
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")
    df["english_name"] = df["english_name"].str.strip()
    df["key"] = df["english_name"].str.lower()
    return df

def split_variants(text):
    # variants may be separated by comma or by |
    return [v.strip() for v in re.split(r"[,|]", text) if v.strip()]

def singular(word):
    # only for comparing words, never to rename
    if word.endswith("oes"):
        return word[:-2]
    if word.endswith("ss") or len(word) <= 3:
        return word
    return word[:-1] if word.endswith("s") else word

def word_key(name):
    # same words in any order, plurals ignored: "potatoes" == "potato"
    return " ".join(sorted(singular(w) for w in name.lower().split()))

def exact_repeats(df):
    return df[df.duplicated("key", keep=False)].sort_values("key")

def same_words_repeats(df):
    d = df.assign(word_key=df["english_name"].map(word_key))
    return d[d["word_key"].duplicated(keep=False) & ~d.duplicated("key", keep=False)]

def near_repeats(df, cutoff=0.9):
    names = sorted(set(df["key"]))
    pairs = []
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if difflib.SequenceMatcher(None, a, b).ratio() >= cutoff:
                pairs.append((a, b))
    return pairs

def variant_clashes(df):
    # one roman variant listed under two or more different english names
    seen = {}
    for _, row in df.iterrows():
        for v in {x.lower() for x in split_variants(row["roman_variants"])}:
            seen.setdefault(v, []).append(row["english_name"])
    return {v: n for v, n in seen.items() if len(set(n)) >= 2}

def repeated_inside_row(df):
    out = []
    for _, row in df.iterrows():
        vs = [x.lower() for x in split_variants(row["roman_variants"])]
        if len(vs) != len(set(vs)):
            out.append((row["english_name"], vs))
    return out

df = load_ingredients(CSV_PATH)
print("rows:", len(df), "| unique english names:", df["key"].nunique())

print("\n1. Exact repeats of english_name (ignoring capitals and spaces):")
ex = exact_repeats(df)
print(ex[["index", "english_name", "roman_variants"]].to_string(index=False) if len(ex) else "   none")

print("\n2. Same words, different order or plural:")
sw = same_words_repeats(df)
print(sw[["english_name", "word_key"]].to_string(index=False) if len(sw) else "   none")

print("\n3. Names that are almost the same (possible typos, similarity >= 0.9):")
nr = near_repeats(df)
print("\n".join(f"   {a!r}  ~  {b!r}" for a, b in nr) if nr else "   none")

print("\n4. One roman variant under 2 or more english names:")
vc = variant_clashes(df)
print("\n".join(f"   {v!r}: {', '.join(sorted(set(n)))}" for v, n in sorted(vc.items())) if vc else "   none")

print("\n5. Same variant written twice inside one row:")
ri = repeated_inside_row(df)
print("\n".join(f"   {n}: {v}" for n, v in ri) if ri else "   none")
